In [1]:
from pathlib import Path
import re
import shutil
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import SUBJECTS, GROUPS, TASK_STAGES
from config.paths import SOURCE_DATA_DIR


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

TASK = "DeCRAT"
TASK_BLOCK = "adaptation"

# True: copy selected STCs
# False: only report which files would be selected
COPY_FILES = True


# ============================================================
# HELPERS
# ============================================================

def remove_hemi_suffix(path):
    """
    Return the STC filename without the hemisphere suffix.

    Example:
        epoch_001-lh.stc -> epoch_001
        epoch_001-rh.stc -> epoch_001
    """
    return re.sub(
        r"-(lh|rh)\.stc$",
        "",
        path.name,
        flags=re.IGNORECASE,
    )


def get_epoch_number(epoch_name):
    """
    Extract the last integer from an STC pair name.

    Examples:
        epoch_001       -> 1
        trial_025       -> 25
        sub34_epoch_143 -> 143
    """
    numbers = re.findall(r"\d+", epoch_name)

    if not numbers:
        raise ValueError(
            f"Could not identify an epoch number in: {epoch_name}"
        )

    return int(numbers[-1])


def find_stc_pairs(stc_folder):
    """
    Find complete paired -lh.stc / -rh.stc epochs.

    Returns
    -------
    pairs : list of dictionaries
        Each item contains:
        {
            "name": common epoch name,
            "number": extracted epoch number,
            "lh": left-hemisphere path,
            "rh": right-hemisphere path
        }
    """
    stc_folder = Path(stc_folder)

    lh_files = {
        remove_hemi_suffix(path): path
        for path in stc_folder.glob("*-lh.stc")
    }

    rh_files = {
        remove_hemi_suffix(path): path
        for path in stc_folder.glob("*-rh.stc")
    }

    lh_names = set(lh_files)
    rh_names = set(rh_files)

    complete_names = lh_names & rh_names
    missing_rh = sorted(lh_names - rh_names)
    missing_lh = sorted(rh_names - lh_names)

    if missing_rh:
        print("  Warning: left STCs without matching right STCs:")
        for name in missing_rh:
            print(f"    {name}-lh.stc")

    if missing_lh:
        print("  Warning: right STCs without matching left STCs:")
        for name in missing_lh:
            print(f"    {name}-rh.stc")

    pairs = []

    for name in complete_names:
        pairs.append(
            {
                "name": name,
                "number": get_epoch_number(name),
                "lh": lh_files[name],
                "rh": rh_files[name],
            }
        )

    # Sort by extracted epoch/trial number
    pairs.sort(key=lambda item: item["number"])

    return pairs


def copy_stc_pair(pair, destination):
    """Copy both hemisphere files for one epoch."""
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)

    shutil.copy2(pair["lh"], destination / pair["lh"].name)
    shutil.copy2(pair["rh"], destination / pair["rh"].name)


# ============================================================
# SPLIT ONE ADAPTATION FOLDER
# ============================================================

def split_adaptation_folder(stc_folder):
    """
    Select the first and final thirds of the available paired
    adaptation epochs.
    """
    stc_folder = Path(stc_folder)

    pairs = find_stc_pairs(stc_folder)
    n_epochs = len(pairs)

    if n_epochs < 3:
        print(f"  Not enough complete STC pairs: {n_epochs}")
        return None

    n_third = n_epochs // 3

    early_pairs = pairs[:n_third]
    late_pairs = pairs[-n_third:]

    early_folder = stc_folder / "early"
    late_folder = stc_folder / "late"

    print(f"  Complete STC pairs: {n_epochs}")
    print(f"  Early epochs:       {len(early_pairs)}")
    print(f"  Late epochs:        {len(late_pairs)}")

    print(
        "  Early epoch range: "
        f"{early_pairs[0]['number']}–{early_pairs[-1]['number']}"
    )
    print(
        "  Late epoch range:  "
        f"{late_pairs[0]['number']}–{late_pairs[-1]['number']}"
    )

    if COPY_FILES:
        for pair in early_pairs:
            copy_stc_pair(pair, early_folder)

        for pair in late_pairs:
            copy_stc_pair(pair, late_folder)

        print(f"  Early output: {early_folder}")
        print(f"  Late output:  {late_folder}")
    else:
        print("  Dry run: no files copied.")

    return {
        "n_total": n_epochs,
        "n_third": n_third,
        "early_pairs": early_pairs,
        "late_pairs": late_pairs,
        "early_folder": early_folder,
        "late_folder": late_folder,
    }


# ============================================================
# RUN FOR BOTH PLAN AND GO
# ============================================================

for group in GROUPS:
    for subject in SUBJECTS[group]:
        for task_stage in TASK_STAGES:

            # Expected structure:
            #
            # SOURCE_DATA_DIR/
            # └── group/
            #     └── subject/
            #         └── plan or go/
            #             └── adaptation/
            #                 ├── epoch_001-lh.stc
            #                 ├── epoch_001-rh.stc
            #                 └── ...

            stc_folder = (
                Path(SOURCE_DATA_DIR)
                / group
                / subject
                / TASK
                / task_stage
                / TASK_BLOCK
            )

            print("\n" + "=" * 70)
            print(
                f"Group: {group} | Subject: {subject} | "
                f"Stage: {task_stage}"
            )
            print(f"Input: {stc_folder}")

            if not stc_folder.exists():
                print("  Folder not found — skipped.")
                continue

            split_adaptation_folder(stc_folder)


Group: Y | Subject: s1_pac_sub01 | Stage: plan
Input: F:\# study 2\eeg_data\epochs\source\Y\s1_pac_sub01\DeCRAT\plan\adaptation
  Complete STC pairs: 135
  Early epochs:       45
  Late epochs:        45
  Early epoch range: 0–44
  Late epoch range:  90–134
  Early output: F:\# study 2\eeg_data\epochs\source\Y\s1_pac_sub01\DeCRAT\plan\adaptation\early
  Late output:  F:\# study 2\eeg_data\epochs\source\Y\s1_pac_sub01\DeCRAT\plan\adaptation\late

Group: Y | Subject: s1_pac_sub01 | Stage: go
Input: F:\# study 2\eeg_data\epochs\source\Y\s1_pac_sub01\DeCRAT\go\adaptation
  Complete STC pairs: 184
  Early epochs:       61
  Late epochs:        61
  Early epoch range: 0–60
  Late epoch range:  123–183
  Early output: F:\# study 2\eeg_data\epochs\source\Y\s1_pac_sub01\DeCRAT\go\adaptation\early
  Late output:  F:\# study 2\eeg_data\epochs\source\Y\s1_pac_sub01\DeCRAT\go\adaptation\late

Group: Y | Subject: s1_pac_sub07 | Stage: plan
Input: F:\# study 2\eeg_data\epochs\source\Y\s1_pac_sub07\D

In [3]:
GROUPS

['Y', 'O']